# Model A
---

Load the preprocessed datasets, vocabulary, config and dataloaders using the following cell.

In [6]:
from utils import load_sets, load_vocab, load_config, get_dataloaders

train_set, val_set, test_set = load_sets()
vocab = load_vocab()
config = load_config()

train_loader, val_loader, test_loader = get_dataloaders(
    train_set, val_set, test_set,
    vocab,
    config['MAX_LENGTH'],
    config['BATCH_SIZE']
)

In [7]:
train_set.head()

,input,response
0,system>perference >appearance i am there,after that click on effects tab and choose 'ef...
1,what is -gt switch in bash? greater? cool,see: man test
2,how to get the calender option for thunderbird...,where to get this package
3,nobody knows how to turn off x server in lubun...,is one way
4,finally... i installed ubuntu... but i have te...,: download it from ubuntu.org


In [8]:
config

{'MAX_LENGTH': 30,
 'MIN_TOKEN_FREQ': 5,
 'MIN_LENGTH': 2,
 'BATCH_SIZE': 64,
 'VOCAB_SIZE': 24788,
 'PAD_IDX': 0,
 'SOS_IDX': 1,
 'EOS_IDX': 2,
 'UNK_IDX': 3}

In [9]:
import torch

# Always use cuda if available (NVIDIA GPU), then try for apple silicon, otherwise just a plain old CPU (but training will be vastly slower)
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f'Using device: {torch.cuda.get_device_name(0)}')
elif torch.backends.mps.is_available():
    device = torch.device("mps")
    print('Using device: MPS (Apple Silicon)')
else:
    device = torch.device("cpu")
    print('Using device: CPU')

Using device: NVIDIA GeForce RTX 4060 Laptop GPU


Quick plan:

1. Define the Encoder  — GRU that reads input, returns ~~all hidden states~~ + final hidden state
2. Define the Decoder  — GRU that generates output token by token using final encoder hidden state
3. Define the Seq2Seq  — wrapper that connects encoder and decoder, handles teacher forcing
4. Training loop       — with validation loss monitoring
5. Inference function  — greedy decoding for generating responses
6. Evaluation          — BLEU score on test set + manual examples

## Encoder

---

The encoder will read the input sequences we've prepared, and will output a single final hidden state to initialise the decoder. So, the architecture is actually quite simple:
- An embedding layer (dataset has low GloVe coverage, so we can train embeddings from scratch).
- A stack of Gated Recurrent Units (combats vanishing gradients, fewer parameters to train than an LSTM, with comparable performance).
- Dropout as a regularisation technique to combat overfitting and improve generalisation (only applied during training). We'll apply dropout between the embedding layer, and between the layers in the GRU.

Defining a sequential model in PyTorch follows the usual pattern:
- Extend the [`nn.Module`](https://docs.pytorch.org/docs/stable/generated/torch.nn.Module.html) base class.
- Initialise your layers in the constructor.
- Override the `forward` method, explicitly feeding X inputs through your layers.

Please read the [GRU](https://docs.pytorch.org/docs/stable/generated/torch.nn.GRU.html) documentation, it explains a lot! :)

In [12]:
import torch
import torch.nn as nn

class Encoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, padding_idx, hidden_dim, num_layers, dropout_proba):
        super().__init__()
        self.embedding = nn.Embedding(
            vocab_size, 
            embed_dim, 
            padding_idx=padding_idx # Pad out to sequence length
        )
        self.gru = nn.GRU(
            embed_dim,
            hidden_dim, # The dimension of the hidden state vector maintained through timesteps
            num_layers=num_layers,
            batch_first=True, # This just moves batch to the first dimension, to match our dataloader implementation
            dropout=dropout_proba if num_layers > 1 else 0 # also apply dropout between GRU layers
        )
        self.dropout = nn.Dropout(dropout_proba)

    def forward(self, X):
        # X: [batch_size, seq_length]
        X = self.embedding(X)
        X = self.dropout(X) # Applying dropout here is standard for seq2seq, it's the largest parameter space in the network
        # X: [batch_size, seq_length, embed_dim]
        _, hidden = self.gru(X)
        # hidden: [num_layers, batch_size, hidden_dim] (our final hidden state)
        
        # We'll ignore the outputs for now, as this model won't use attention
        return hidden

## Decoder

---

The decoder must first initialise its hidden state with the final output hidden state of the encoder. Then, at each timestep (**a forward pass through the network processing one token**):
1. Obtain the embedding for the previous token
2. Pass it through the GRU
3. Project the output to the vocabulary size to predict the next token (using a simple fully connected linear layer)

Outside of this, its layer initialisation is very similar to the encoder. 

Note we're also training separate embeddings in the decoder rather using shared embeddings between the encoder and decoder (called _weight tying_ -- possible note in the report). This is to keep the embeddings specialised as the encoder and decoder objectives are fundamentally different: the encoder must encode meaning into the hidden state, the decoder must utilise this state to predict the next token given the previous one. Therefore, allowing both to represent tokens in a way that optimises their objectives is the intent.

In [14]:
class Decoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, padding_idx, hidden_dim, num_layers, dropout_proba):
        super().__init__()
        self.embedding = nn.Embedding(
            vocab_size, 
            embed_dim, 
            padding_idx=padding_idx
        )
        self.gru = nn.GRU(
            embed_dim,
            hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout_proba if num_layers > 1 else 0
        )
        self.linear = nn.Linear(hidden_dim, vocab_size) # Simply project from the hidden dimension size down to the vocabulary size 
        self.dropout = nn.Dropout(dropout_proba)

    def forward(self, X, hidden):
        # X: [batch_size] (per sample, just one token)
        X = X.unsqueeze(1)
        # X: [batch_size, 1] (add seq_length dimension, which is just 1, since we're only processing one token at a time)
        X = self.embedding(X)
        X = self.dropout(X)
        # X: [batch_size, 1, embed_dim]
        output, hidden = self.gru(X, hidden)
        # output: [batch_size, 1, hidden_dim] (?)
        # hidden: [num_layers, batch_size, hidden_dim] (the next hidden state to use at the next timestep)

        pred = self.linear(output.squeeze(1)) # Convert back to [batch_size] and project to vocabulary
        # pred: [batch_size, vocab_size] (this is our next token prediction!)

        return pred, hidden

## Seq2Seq Wrapper

---

This is the wrapper model that will be built from the encoder and decoder. It will:
1. Pass the full input sequence through the encoder to produce a final hidden state
2. Use this final hidden state to initialise the decoder
3. Loop through each decoding timestep and feed each token in one at a time
4. Apply **teacher forcing** during training only (feeding the ground truth labels in at each timestep to improve stability and reduce _exposure bias_I)

The variables passed into the `forward` method are those we prepared at the data preparation stage:
- `encoder_input`: the encoded input sequence (for the encoder)
- `decoder_input`: the encoded response with SOS prepended

In [22]:
class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, device):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device

    def forward(self, encoder_input, decoder_input, teacher_forcing_proba=0.5):
        # encoder_input: [batch_size, encoder_input_length]
        # decoder_input: [batch_size, decoder_input_length]
        
        batch_size = encoder_input.shape[0]
        decoder_input_length = decoder_input.shape[1]
        vocab_size = self.decoder.linear.out_features

        # A tensor for storing the decoder predictions (ensure tensor is on the same device as the model)
        outputs = torch.zeros(batch_size, decoder_input_length, vocab_size).to(self.device)

        # Forward pass through the encoder to obtain the final hidden state
        hidden = self.encoder(encoder_input)

        # First decoder input is the <SOS> token for every sample
        next_decoder_input = decoder_input[:, 0]

        # Every timestep: every token in the sequence, produce a prediction
        for timestep in range(1, decoder_input_length):
            pred, hidden = self.decoder(next_decoder_input, hidden) # Use the next decoder input and last hidden state to produce the next token prediction
            outputs[:, timestep, :] = pred # For all samples (tokens) in the batch, assign all decoder predictions (probability distributions) at this timestep

            # Teacher forcing (using the ground truth or model prediction with highest probability)
            # We'll apply teacher forcing with the given probability. This probability will balance learning 
            # from ground truth and learning from experience with the end goal of improving ability to generalise on unseen data (during inference)
            use_teacher_forcing = torch.rand(1).item() < teacher_forcing_proba
            next_decoder_input = decoder_input[:, timestep] if use_teacher_forcing else pred.argmax(1) # ground truth OR best prediction

        return outputs

## Training Loop

---

This is the main training loop, where we'll use cross entropy loss, Adam optimiser, and gradient clipping to manage exploding gradients (this is common in BPTT).

We can use a `max_norm=1.0` to scale the gradient vector down if its norm exceeds 1 to help keep the weight updates stable.

First things first: let's define the training and model hyperparameters, and instantiate the model.

In [25]:
# Training Parameters
EPOCHS = 5
CLIP_MAX_NORM = 1.0
TEACHER_FORCING_PROBA = 0.5

# Model Hyperparameters
ENCODER_EMBED_DIM = 128
ENCODER_HIDDEN_DIM = 256
ENCODER_NUM_LAYERS = 2
ENCODER_DROPOUT_PROBA = 0.3

DECODER_EMBED_DIM = 128
DECODER_HIDDEN_DIM = 256
DECODER_NUM_LAYERS = 2
DECODER_DROPOUT_PROBA = 0.3

model_a = Seq2Seq(
    encoder=Encoder(
        vocab_size=config['VOCAB_SIZE'], 
        embed_dim=ENCODER_EMBED_DIM, 
        padding_idx=config['PAD_IDX'], 
        hidden_dim=ENCODER_HIDDEN_DIM,
        num_layers=ENCODER_NUM_LAYERS,
        dropout_proba=ENCODER_DROPOUT_PROBA
    ),
    decoder = Decoder(
        vocab_size=config['VOCAB_SIZE'],
        embed_dim=DECODER_EMBED_DIM,
        padding_idx=config['PAD_IDX'],
        hidden_dim=DECODER_HIDDEN_DIM,
        num_layers=DECODER_NUM_LAYERS,
        dropout_proba=DECODER_DROPOUT_PROBA
    ),
    device=device
).to(device) # Move model to device

In [35]:
#!mkdir -p models/model_a
from pathlib import Path

Path("models/model_a").mkdir(parents=True, exist_ok=True)

In [37]:
from torch.optim import Adam
from pathlib import Path
import time

checkpoint_dir = Path('models/model_a')
model_a_baseline_state_dict = checkpoint_dir / 'model_a_baseline.pt'

criterion = nn.CrossEntropyLoss(ignore_index=config['PAD_IDX'])
optimiser = Adam(model_a.parameters(), lr=1e-3)

# Latest checkpoint to resume from
existing_checkpoints = sorted(checkpoint_dir.glob('checkpoint_epoch_*.pt'))

if model_a_baseline_state_dict.exists():
    print('Loading baseline model weights...')
    model_a.load_state_dict(torch.load(model_a_baseline_state_dict, weights_only=True))
    start_epoch = EPOCHS
elif existing_checkpoints:
    latest = existing_checkpoints[-1]
    checkpoint = torch.load(latest, weights_only=True)
    model_a.load_state_dict(checkpoint['model_state_dict'])
    optimiser.load_state_dict(checkpoint['optimiser_state_dict'])
    start_epoch = len(existing_checkpoints)
    print(f'Resuming from epoch {start_epoch}/{EPOCHS}')
else:
    print('No checkpoints, training model from scratch...')
    start_epoch = 0
    
def train_epoch(model, loader, optimiser, criterion, device, clip_max_norm, teacher_forcing_proba):
    model.train() # Enables training mode (includes enabling dropout)
    total_loss = 0
    total_batches = len(loader)
    log_interval = max(1, int(total_batches * 0.01)) # Every 10%

    for batch_idx, (encoder_input_batch, decoder_input_batch, decoder_target_batch) in enumerate(loader):
        # Remember tensors and models must be on the same device
        encoder_input_batch = encoder_input_batch.to(device)
        decoder_input_batch = decoder_input_batch.to(device)
        decoder_target_batch = decoder_target_batch.to(device)

        optimiser.zero_grad()
        predictions = model(encoder_input_batch, decoder_input_batch, teacher_forcing_proba=teacher_forcing_proba) # Forward pass
        predictions = predictions.permute(0, 2, 1) # Need to reshape for CrossEntropyLoss, this just reorders dimensions
        loss = criterion(predictions, decoder_target_batch) # Compute loss
        loss.backward() # BPTT
        torch.nn.utils.clip_grad_norm_(model.parameters(), clip_max_norm) # Gradient clipping comes before weight updates
        optimiser.step() # Update weights
        total_loss += loss.item() 

        # Log every 25% of batches (or whatever log_percent you set)
        if (batch_idx + 1) % log_interval == 0:
            avg_loss = total_loss / (batch_idx + 1)
            progress = 100 * (batch_idx + 1) / total_batches
            print(f'  [{progress:.0f}%] Batch {batch_idx+1}/{total_batches} | Avg Loss: {avg_loss:.4f}')

    return total_loss / len(loader) # Average loss per batch

def val_epoch(model, loader, criterion, device):
    model.eval() # Deactivates dropout
    total_loss = 0
    with torch.no_grad(): # Disable gradient computation and tracking
        for encoder_input_batch, decoder_input_batch, decoder_target_batch in loader:
            encoder_input_batch = encoder_input_batch.to(device)
            decoder_input_batch = decoder_input_batch.to(device)
            decoder_target_batch = decoder_target_batch.to(device)

            # During evaluation there should be no teacher forcing
            predictions = model(encoder_input_batch, decoder_input_batch, teacher_forcing_proba=0.0)
            predictions = predictions.permute(0, 2, 1)
            loss = criterion(predictions, decoder_target_batch)
            total_loss += loss.item()

    return total_loss / len(loader)

# Main training loop
epoch_times = []
for epoch in range(start_epoch, EPOCHS):
    start = time.time()
    
    train_loss = train_epoch(
        model_a, 
        train_loader, 
        optimiser, 
        criterion, 
        device, 
        CLIP_MAX_NORM,
        TEACHER_FORCING_PROBA
    )
    val_loss = val_epoch(
        model_a, 
        val_loader, 
        criterion, 
        device
    )
    elapsed = time.time() - start
    epoch_times.append(elapsed)
    
    print(f'Epoch {epoch+1}/{EPOCHS} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Time: {elapsed:.2f}s')

    # Checkpoint
    torch.save({
        'model_state_dict': model_a.state_dict(), # The actual model and its weights
        'optimiser_state_dict': optimiser.state_dict(), #
    }, checkpoint_dir / f'checkpoint_epoch_{epoch+1}.pt')

# Save model's state_dict as per https://docs.pytorch.org/tutorials/beginner/saving_loading_models.html#saving-loading-model-for-inference
torch.save(model_a.state_dict(), model_a_baseline_state_dict)

# Print benchmarks
if epoch_times:
    print(f'\n--- Benchmark Summary ---')
    print(f'Device: {device}')
    print(f'Total training time: {sum(epoch_times):.2f}s')
    print(f'Average epoch time: {sum(epoch_times)/len(epoch_times):.2f}s')
    print(f'Fastest epoch: {min(epoch_times):.2f}s')
    print(f'Slowest epoch: {max(epoch_times):.2f}s')
else:
    print('Training already complete!')

No checkpoints, training model from scratch...
  [1%] Batch 27/2741 | Avg Loss: 6.4941
  [2%] Batch 54/2741 | Avg Loss: 6.5232
  [3%] Batch 81/2741 | Avg Loss: 6.5286
  [4%] Batch 108/2741 | Avg Loss: 6.5371
  [5%] Batch 135/2741 | Avg Loss: 6.5346
  [6%] Batch 162/2741 | Avg Loss: 6.5372
  [7%] Batch 189/2741 | Avg Loss: 6.5434
  [8%] Batch 216/2741 | Avg Loss: 6.5445
  [9%] Batch 243/2741 | Avg Loss: 6.5476
  [10%] Batch 270/2741 | Avg Loss: 6.5470
  [11%] Batch 297/2741 | Avg Loss: 6.5499
  [12%] Batch 324/2741 | Avg Loss: 6.5505
  [13%] Batch 351/2741 | Avg Loss: 6.5498
  [14%] Batch 378/2741 | Avg Loss: 6.5489
  [15%] Batch 405/2741 | Avg Loss: 6.5498
  [16%] Batch 432/2741 | Avg Loss: 6.5488
  [17%] Batch 459/2741 | Avg Loss: 6.5488
  [18%] Batch 486/2741 | Avg Loss: 6.5488
  [19%] Batch 513/2741 | Avg Loss: 6.5478
  [20%] Batch 540/2741 | Avg Loss: 6.5487
  [21%] Batch 567/2741 | Avg Loss: 6.5490
  [22%] Batch 594/2741 | Avg Loss: 6.5471
  [23%] Batch 621/2741 | Avg Loss: 6.5465